# Local orchestration + service-hosted ALE execution

This is the minimal ALE counterpart of `eval_service_demo_eog.ipynb`:

1. We provide an offline plan.
2. Local Codex spawns the named specialists.
3. Both specialists work in one service-hosted ALE filesystem through `ale_sandbox`.
4. The service runs the unchanged official evaluator.

`CommandAgent` and `acp_codex_agent` are not used. This is a one-task diagnostic dry run, not a leaderboard run.


## Setup

Set `EVAL_SERVICE_API_KEY` first. The local `codex` CLI uses its existing login (or your own `OPENAI_API_KEY`).


In [ ]:
# Install/update the authenticated SDK if needed:
# WHEEL=$(curl -fsSL -H "Authorization: Bearer $EVAL_SERVICE_API_KEY" "$EVAL_SERVICE_URL/sdk" | python -c "import sys,json; print(json.load(sys.stdin)['path'])")
# curl -fsSL -H "Authorization: Bearer $EVAL_SERVICE_API_KEY" "${EVAL_SERVICE_URL}${WHEEL}" -o "/tmp/${WHEEL##*/}"
# pip install --upgrade --force-reinstall --no-deps "/tmp/${WHEEL##*/}"

import json, os, shutil, subprocess, tempfile
from pathlib import Path

from simple_agentic_evals import EvalClient
from simple_agentic_evals.command_agent import materialize_task_workspace

SERVICE_URL = os.environ.get(
    "EVAL_SERVICE_URL", "https://educator-marrow-cultural.ngrok-free.dev"
)
if not os.environ.get("EVAL_SERVICE_API_KEY"):
    raise RuntimeError("Set EVAL_SERVICE_API_KEY first")
if not shutil.which("codex"):
    raise RuntimeError("Install and authenticate the local codex CLI first")

client = EvalClient(base_url=SERVICE_URL, timeout=1800)
MODEL = "gpt-5"


## Offline plan

The plan is fixed before evaluation, just like the parsed plan in the EOG notebook. Writes are sequential because both specialists share the same output directory.


In [ ]:
PLAN = [
    {
        "agent": "numerics_stats",
        "instruction": (
            "Read base/input/problem_spec.md, implement the American-option solver, "
            "and write base/output/results.json plus "
            "base/output/exercise_boundary_tier2.npy. Validate the values."
        ),
    },
    {
        "agent": "runtime_base",
        "instruction": (
            "Independently read the specification and audit the two hosted output files. "
            "Rerun schema and consistency checks, and correct the files if necessary."
        ),
    },
]
PLAN


## Run the local orchestrator and grade

The workspace helper only installs the selected task-scoped agent definitions and MCP configuration. Local Codex still owns orchestration; the eval service owns the shared sandbox and grading. The task context manager cleans up the session on success or failure.


In [ ]:
task = client.task(
    "evovling_agents", "ale", "full",
    "business_finance/american_option_pricing_ls",
    split="test", resource_mode="accumulative",
)

with task:
    task.start_remote_sandbox(timeout=900)
    run_dir = Path(tempfile.mkdtemp(prefix="ale-local-plan-"))
    workspace = materialize_task_workspace(
        task, run_dir, codex=True, fetch_sandbox_inputs=False,
        remote_mcp_server=task.remote_mcp_server,
    )

    prompt = f"""
{task.system_prompt}

{task.user_prompt}

The planning step is already complete. Execute this fixed plan in order:
{json.dumps(PLAN, indent=2)}

Spawn each named specialist, wait for it, and pass its findings to the next one.
Every terminal and filesystem action must use the `ale_sandbox` MCP. Do not use local
input/output directories. Finish only after the second specialist confirms both outputs.
""".strip()

    proc = subprocess.run(
        [
            "codex", "exec", "--ephemeral", "--ignore-rules",
            "--json", "--skip-git-repo-check",
            "--sandbox", "workspace-write",
            "-c", "sandbox_workspace_write.network_access=true",
            "-c", "model_reasoning_effort=\"high\"",
            "-m", MODEL, "-",
        ],
        input=prompt, text=True, cwd=workspace["root"],
        capture_output=True, timeout=7200, check=False,
    )

    # Keep a redacted local orchestration trace.
    trace, error = proc.stdout, proc.stderr
    for secret_name in ("EVAL_SERVICE_API_KEY", "OPENAI_API_KEY"):
        secret = os.environ.get(secret_name, "")
        if secret:
            trace = trace.replace(secret, "[REDACTED]")
            error = error.replace(secret, "[REDACTED]")
    (run_dir / "codex.jsonl").write_text(trace, encoding="utf-8")
    if proc.returncode:
        raise RuntimeError(f"Codex exited {proc.returncode}: {error[-2000:]}")

    grade = task.grade(keep_alive=True)
    print("trace:", run_dir / "codex.jsonl")
    print(
        f"score={grade.pass_rate:.3f} success={grade.overall_success} "
        f"execution={grade.execution_mode}"
    )
    for verifier in grade.per_verifier or []:
        print(verifier.get("name"), verifier.get("score"), verifier.get("passed"))


The essential difference from EOG is only the shared state: EOG nodes mutate a hosted database through business tools; ALE specialists mutate a hosted filesystem through `ale_sandbox`. The local offline plan and official final grade remain the same pattern.
